# Reflective MCTS (R-MCTS) | Advanced Planning & Search

In [1]:
import math
import re
from dataclasses import dataclass, field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
@dataclass
class Node:
    state: str
    parent: Optional['Node'] = None
    children: List['Node'] = field(default_factory=list)
    visits: int = 0; value: float = 0.0

    def ucb1(self, c=1.41):
        if self.visits == 0: return float('inf')
        return (self.value / self.visits) + c * math.sqrt(math.log(self.parent.visits) / self.visits)

def select(n):
    while n.children: n = max(n.children, key=lambda c: c.ucb1())
    return n

def expand(node, problem, wisdom=""):
    ctx = f"\nWisdom from prior episode:\n{wisdom}" if wisdom else ""
    resp = model.invoke(
        f"Problem: {problem}\nPartial solution:\n{node.state}{ctx}\n\n"
        "Propose 3 next steps (numbered 1-3, one line each)."
    )
    for line in resp.content.strip().splitlines():
        step = re.sub(r"^\d+[\.)\s]", "", line.strip())
        if step: node.children.append(Node(state=f"{node.state}\n- {step}", parent=node))

def simulate(node, problem):
    resp = model.invoke(
        f"Problem: {problem}\nPartial solution:\n{node.state}\n\n"
        "Score 0.0-1.0 how promising this is. Reply with ONLY a number."
    )
    try: return max(0.0, min(1.0, float(re.search(r"[\d.]+", resp.content).group())))
    except: return 0.5

def backprop(n, v):
    while n: n.visits += 1; n.value += v; n = n.parent

def run_episode(problem, iterations=8, wisdom=""):
    root = Node(state="Start")
    scores = []
    wisdom_used_at = None  # track which expansion first used wisdom
    for i in range(iterations):
        leaf = select(root)
        if leaf.visits > 0 and not leaf.children:
            expand(leaf, problem, wisdom)
            if wisdom and wisdom_used_at is None:
                wisdom_used_at = i + 1
            leaf = leaf.children[0] if leaf.children else leaf
        s = simulate(leaf, problem)
        backprop(leaf, s)
        scores.append(s)
    # Collect all leaf states for reflection
    leaves = []
    def gather(n):
        if not n.children: leaves.append((n.state, n.value / max(n.visits, 1)))
        for c in n.children: gather(c)
    gather(root)
    best_path_score = max(s for _, s in leaves) if leaves else 0.0
    return scores, sorted(leaves, key=lambda x: -x[1]), best_path_score, wisdom_used_at

def reflect(top_leaves, bottom_leaves):
    resp = model.invoke(
        f"Best paths:\n" + "\n".join(f"[{s:.2f}] {p}" for p, s in top_leaves[:3]) +
        f"\n\nWorst paths:\n" + "\n".join(f"[{s:.2f}] {p}" for p, s in bottom_leaves[:3]) +
        "\n\nAnswer these 3 questions:\n"
        "1. What strategies worked in the best paths?\n"
        "2. What failed in the worst paths?\n"
        "3. Give 3 specific rules for the next episode to follow."
    )
    return resp.content.strip()

In [5]:
# --- Run 2 episodes with cross-episode reflection ---
problem = "Solve step by step: if a train travels 120km in 1.5 hours, what is its speed in m/s?"

scores1, leaves1, best1, _ = run_episode(problem, iterations=8)
wisdom = reflect(leaves1[:3], leaves1[-3:])
print(f"Episode 1 best path score: {best1:.2f}")
print(f"\nWisdom generated:\n{wisdom}\n")

Episode 1 best path score: 1.00

Wisdom generated:
1. **What strategies worked in the best paths?**
   - The best paths focused on clear, step-by-step conversions of units to ensure consistent measurement systems for subsequent calculations. Specifically, they involved:
     - Accurately converting kilometers to meters to standardize the unit of distance.
     - Converting hours to seconds to standardize the unit of time.
     - These conversions laid the groundwork for accurate calculation of speed, ensuring all variables were in compatible units (meters and seconds).

2. **What failed in the worst paths?**
   - The worst paths demonstrated a lack of necessary conversions before attempting the final calculations. Issues included:
     - Jumping directly to calculating speed without first converting time to the appropriate unit (seconds), which can lead to incorrect results.
     - Focusing on division or further calculations before ensuring all input quantities were standardized to th

In [6]:
scores2, leaves2, best2, wisdom_iter = run_episode(problem, iterations=8, wisdom=wisdom)
print(f"Episode 2 best path score: {best2:.2f}")
if wisdom_iter:
    print(f"Wisdom first applied at: iteration {wisdom_iter} of episode 2")
print(f"\n--- Cross-Episode Comparison ---")
print(f"Episode 1 best path score: {best1:.2f} | Episode 2 best path score: {best2:.2f} | "
      f"Delta: {best2 - best1:+.2f}")

Episode 2 best path score: 0.90
Wisdom first applied at: iteration 2 of episode 2

--- Cross-Episode Comparison ---
Episode 1 best path score: 1.00 | Episode 2 best path score: 0.90 | Delta: -0.10
